# LangChain: Embedding Concepts

## Outline
* Embedding concept (semantic vectorization)
* Setting up `OpenAIEmbeddings` in LangChain
* Difference between `embed_query` and `embed_documents`
* Manual semantic similarity calculation (Cosine Similarity) without a Vector Store to understand the underlying mathematics


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model

_ = load_dotenv(find_dotenv())

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0, api_key=api_key, base_url=base_url)
ollama = init_chat_model("llama3.1:8b ", model_provider="ollama", temperature=0)

## 1. Embedding Concept

In modern natural language processing approaches, words and sentences are converted into arrays of numbers (vectors) that represent their semantic meaning in a multidimensional space. Texts with similar meanings will be closer to each other in this space.


In [2]:
# pip install langchain-openai numpy
from langchain_openai import OpenAIEmbeddings
from sklearn.metrics.pairwise import cosine_similarity


# Initialize the embedding model using LangChain
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", api_key=api_key, base_url=base_url)
print("Embedding model initialized successfully!")


Embedding model initialized successfully!


## 2. The `embed_documents` Method

This method is used to convert a **list of documents** (the content of your catalog database) into vectors. Its output will be a list of lists.


In [3]:
# A small corpus of documents to teach similarity
documents = [
    "Artificial intelligence is changing the world.",
    "Artificial intelligence technology is transforming human life.",
    "Ali's driving is good.",
    "The air is dirty.",
    "The air is polluted."
]

# Convert multiple documents into embedding vectors
doc_vectors = embeddings.embed_documents(documents)

print(f"Total documents embedded: {len(doc_vectors)}")
print(f"Dimensions of first document: {len(doc_vectors[0])}")


Total documents embedded: 5
Dimensions of first document: 3072


In [4]:
cos_sim_matrix = cosine_similarity(doc_vectors)
print(cos_sim_matrix)

[[1.         0.80841626 0.17809494 0.15380713 0.15957583]
 [0.80841626 1.         0.15141876 0.10547805 0.12954573]
 [0.17809494 0.15141876 1.         0.12796467 0.11373077]
 [0.15380713 0.10547805 0.12796467 1.         0.88709598]
 [0.15957583 0.12954573 0.11373077 0.88709598 1.        ]]


## 3. Manual Cosine Similarity Calculation

To understand how vector databases (such as FAISS) work, let's mathematically compare the user's query vector with the document vectors using the Cosine Similarity formula.


In [6]:
import numpy as np

def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors manually."""
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    return dot_product / (norm_v1 * norm_v2)

text = "machine learning and deep learning"
query_vector = embeddings.embed_query(text)
print(f"Query: '{text}'\n")
print("Manual Search Results via Math:")

# Compare query vector against all document vectors
for i, doc_vec in enumerate(doc_vectors):
    score = cosine_similarity(query_vector, doc_vec)
    print(f"  Document {i+1}: \"{documents[i]}\"")
    print(f"  Similarity Score: {score * 100:.2f}%\n")


Query: 'machine learning and deep learning'

Manual Search Results via Math:
  Document 1: "Artificial intelligence is changing the world."
  Similarity Score: 39.14%

  Document 2: "Artificial intelligence technology is transforming human life."
  Similarity Score: 36.78%

  Document 3: "Ali's driving is good."
  Similarity Score: 11.10%

  Document 4: "The air is dirty."
  Similarity Score: 4.96%

  Document 5: "The air is polluted."
  Similarity Score: 4.83%



In [7]:
text = "there is an air polution"
query_vector = embeddings.embed_query(text)
print(f"Query: '{text}'\n")
print("Manual Search Results via Math:")

# Compare query vector against all document vectors
for i, doc_vec in enumerate(doc_vectors):
    score = cosine_similarity(query_vector, doc_vec)
    print(f"  Document {i+1}: \"{documents[i]}\"")
    print(f"  Similarity Score: {score * 100:.2f}%\n")


Query: 'there is an air polution'

Manual Search Results via Math:
  Document 1: "Artificial intelligence is changing the world."
  Similarity Score: 14.61%

  Document 2: "Artificial intelligence technology is transforming human life."
  Similarity Score: 12.02%

  Document 3: "Ali's driving is good."
  Similarity Score: 11.02%

  Document 4: "The air is dirty."
  Similarity Score: 68.69%

  Document 5: "The air is polluted."
  Similarity Score: 76.72%

